###RAGpipelines-Data Ingestion to vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shiva\AppData\Local\Temp\ipykernel_14400\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
c:\Users\shiva\OneDrive\Desktop\RAGpipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" loaded{len(documents)} pages")
        except Exception as e:
            print(f" error:{e}")
    print(f"\n total documents loaded: {len(all_documents)}")
    return all_documents

all_pdfs_documents=process_all_pdfs("../data")

Found 3 PDF files to process

Processing:100 Endgames You Must Know.pdf
 loaded493 pages

Processing:final research.pdf
 loaded6 pages

Processing:Shiva_resume.pdf
 loaded1 pages

 total documents loaded: 500


In [3]:
all_pdfs_documents

[Document(metadata={'producer': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creator': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creationdate': '2016-08-23T00:16:30+00:00', 'author': 'Jesus de la Villa', 'title': '100 Endgames You Must Know: Vital Lessons for Every Chess Player Improved and Expanded', 'moddate': '2016-08-22T21:46:14-04:00', 'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf', 'total_pages': 493, 'page': 0, 'page_label': '1', 'source_file': '100 Endgames You Must Know.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creator': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creationdate': '2016-08-23T00:16:30+00:00', 'author': 'Jesus de la Villa', 'title': '100 Endgames You Must Know: Vital Lessons for Every Chess Player Improved and Expanded', 'moddate': '2016-08-22T21:46:14-04:00', 'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf', 'total_pages': 493, 'page': 1, 'page_label':

In [4]:
###text splitting into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n example chunk:")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"MetaData: {split_docs[0].metadata}")
    return split_docs

In [5]:
chunks=split_documents(all_pdfs_documents)
chunks

split 500 documents into 713 chunks

 example chunk:
content: 100 Endgames You Must Know...
MetaData: {'producer': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creator': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creationdate': '2016-08-23T00:16:30+00:00', 'author': 'Jesus de la Villa', 'title': '100 Endgames You Must Know: Vital Lessons for Every Chess Player Improved and Expanded', 'moddate': '2016-08-22T21:46:14-04:00', 'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf', 'total_pages': 493, 'page': 1, 'page_label': '2', 'source_file': '100 Endgames You Must Know.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creator': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creationdate': '2016-08-23T00:16:30+00:00', 'author': 'Jesus de la Villa', 'title': '100 Endgames You Must Know: Vital Lessons for Every Chess Player Improved and Expanded', 'moddate': '2016-08-22T21:46:14-04:00', 'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf', 'total_pages': 493, 'page': 1, 'page_label': '2', 'source_file': '100 Endgames You Must Know.pdf', 'file_type': 'pdf'}, page_content='100 Endgames You Must Know'),
 Document(metadata={'producer': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creator': 'calibre 2.63.0 [https://calibre-ebook.com]', 'creationdate': '2016-08-23T00:16:30+00:00', 'author': 'Jesus de la Villa', 'title': '100 Endgames You Must Know: Vital Lessons for Every Chess Player Improved and Expanded', 'moddate': '2016-08-22T21:46:14-04:00', 'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf', 'total_pages': 493

Embedding and VectorstoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:

    def __init__(self, model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    ## to load the model
    def _load_model(self):
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts:List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("Model Not Found")
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts,show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2059.92it/s]


Model loaded successfully. Embedding dimensions: 384


In [8]:
#vectorstore
class Vectorstore:
    def __init__(self, collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"My resume"}
            )
            print(f"vector store initialized. collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing  vector store: {e}")
            raise
    def add_documents(self, documents: List[Any],embeddings: np.ndarray):

        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to the vectore store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):

            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding  documents to vector store :{e}")
            raise
vector_store=Vectorstore()
vector_store

vector store initialized. collection: pdf_documents
Existing documents in collection: 897


In [9]:
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embeddings)

generating embeddings for 713 texts...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches: 100%|██████████| 23/23 [01:10<00:00,  3.08s/it]


generated embeddings with shape: (713, 384)
Adding 713 documents to the vectore store
successfully added 713 documents to vector store
total documents in collection: 1610


retriever pipeline from vectorstore


In [10]:
class RAGRetriever:

    def __init__(self,vector_store:Vectorstore, embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retriever(self,query:str,top_k:int =5,score_threshold: float=0.0)->List[Dict[str,Any]]:
        print(f"Retrieving documents for query : '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")

        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]


                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score=1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("no document found")
            return retrieved_docs
        except Exception as e:
            print(f"error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [11]:
rag_retriever

In [12]:
rag_retriever.retriever("The relative importance of the endgame")

Retrieving documents for query : 'The relative importance of the endgame'
Top k: 5, score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.24it/s]

generated embeddings with shape: (1, 384)
retrieved 5 documents (after filtering)


[{'id': 'doc_e560fae0_24',
  'content': 'endgame themes will be necessary here, but most are learned from the study of basic positions.\nThe \n‘exact endings’\n we need to remember are just a few. Besides, some are really easy to\nmemorise, and others could be considered as marginal in view of their comparative rarity in\npractice. They are just a few, but you must know them well. This fundamental knowledge and the\nconfidence we acquire with it is the starting point to study other positions of greater complexity\nor to turn a technical advantage into victory.\nAfter we have acquired a good command of the basic endings comes the third phase. In it, we',
  'metadata': {'total_pages': 493,
   'creator': 'calibre 2.63.0 [https://calibre-ebook.com]',
   'page': 9,
   'source_file': '100 Endgames You Must Know.pdf',
   'doc_index': 24,
   'producer': 'calibre 2.63.0 [https://calibre-ebook.com]',
   'author': 'Jesus de la Villa',
   'source': '..\\data\\pdf\\100 Endgames You Must Know.pdf',


integration vectordb context pipeline with llm output


In [ ]:
###simple RAG pipeline with groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile", temperature=0.1,max_tokens=1024)


def RAG_simple(query,retriever,llm,top_k=3):
    results=retriever.retriever(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "no relevant  context found"
    prompt=f"""use the following context to answer the question concisely.
      context: {context} 
      query: {query}
      answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [14]:
answer=RAG_simple("give me authors of The next prediction is of vehicle damage localization and The estimation of severity with Deep learning. project",rag_retriever,llm)
print(answer)

Retrieving documents for query : 'give me authors of The next prediction is of vehicle damage localization and The estimation of severity with Deep learning. project'
Top k: 3, score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.55it/s]

generated embeddings with shape: (1, 384)
retrieved 3 documents (after filtering)


The authors of the project are:

1. HEMALATHA
2. A. RAMESH
3. K. SHIVA
4. S. NASREEN
5. V. THRISHA


enchanced ragpipeline


In [16]:
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):

    results=retriever.retriever(query,top_k=top_k,score_threshold=min_score)

    if not results:
        return {'answer':'no relevant context found.','sources':[],'confidence':0.0,'context':''}
    
    context="\n\n".join([doc['content'] for doc in results])

    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:120]+'...'
    } for doc in results]
    confidence=max([doc['similarity_score'] for doc in results])

    prompt=f"""use the following context to answer  the question concisely. \ncontext: \n{context}\n\n question: {query}\n\n answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])

    output={
        'answer':response.content,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context']=context
    return output


result=rag_advanced("give me authors of The next prediction is of vehicle damage localization and The estimation of severity with Deep learning. project",rag_retriever,llm,top_k=3,min_score=0.1,return_context=True)
print("Answer:",result['answer'])
print("sources",result['sources'])
print("confidence",result['confidence'])
print("context preview",result['context'][:120])

Retrieving documents for query : 'give me authors of The next prediction is of vehicle damage localization and The estimation of severity with Deep learning. project'
Top k: 3, score threshold: 0.1
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.80it/s]

generated embeddings with shape: (1, 384)
retrieved 3 documents (after filtering)


Answer: The authors of the project are:

1. HEMALATHA
2. A. RAMESH
3. K. SHIVA
4. S. NASREEN
5. V. THRISHA
sources [{'source': 'final research.pdf', 'page': 0, 'score': 0.4319115877151489, 'preview': 'The next prediction is of vehicle damage localization and  \nThe estimation of severity with Deep learning.  \n \n 1A. HEMA...'}, {'source': 'final research.pdf', 'page': 0, 'score': 0.4319115877151489, 'preview': 'The next prediction is of vehicle damage localization and  \nThe estimation of severity with Deep learning.  \n \n 1A. HEMA...'}, {'source': 'final research.pdf', 'page': 1, 'score': 0.3941311836242676, 'preview': 'methods though simple to realize, they could not be \ntrusted in other light, angles of view, or designs of \nvehicles, th...'}]
confidence 0.4319115877151489
context preview The next prediction is of vehicle damage localization and  
The estimation of severity with Deep learning.  
 
 1A. HEMA
